In [2]:
%load_ext autoreload
%autoreload 2
# %cd /pscratch/sd/b/brenthu/chem_llm
%cd /nfs/roberts/project/pi_vsb4/byh2/chem_llm

import json
import os
import sys
import config
from dotenv import load_dotenv
load_dotenv()
    
# HF_HOME must be set before transformers is imported so it picks up the cache dir.
os.environ["HF_HOME"] = config.HF_HOME
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from agent_core import run_agent

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/nfs/roberts/project/pi_vsb4/byh2/chem_llm


In [3]:
os.makedirs(config.WORK_DIR, exist_ok=True)
os.chdir(config.WORK_DIR)
print(f"Working directory: {os.getcwd()}")

if config.PROJECT_ROOT not in sys.path:
    sys.path.insert(0, config.PROJECT_ROOT)

Working directory: /nfs/roberts/project/pi_vsb4/byh2/chem_llm/test10


In [4]:
# LOAD MODEL

if not config.HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is not set. Run `export HF_TOKEN=hf_xxx` before launching main.py."
    )
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME, token=config.HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
    token=config.HF_TOKEN,
)

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

In [8]:
TASK = """
Write two new Python scripts based off of the pre-existing generate_structures.py and setup_jobs.py that generates perturbed TiO2 crystal structures using pymatgen and input files for quantum espresso.

Required workflow:
1. Read the existing generate_structures.py and note down how each constant and function works as well as anything you don't need for TiO2 (for example, interpolating between phase transitions)
2. Use the generate_cif tool to obtain the base rutile TiO4 crystal structure.
3. In the python script, use pymatgen to generate 50 additional structures for 51 total structures.
4. The perturbations should be chemically reasonable random displacements of the Ti and O atoms at room temperature.
5. Then read the existing setup_jobs.py, noting down what it does and any changes you must make to work with the new workflow
6. Use the get_pseudopotential tool to create your own template folder. Please use norm conserving, fully relativisitc, pbesol, stringent pseudopotentials
6. Then write a new python file. Leave SLURM account, email, etc. blank

Requirements:
- Base the perturbations on the endpoint structure obtained from generate_cif.
- Organize the output files in a clear directory structure.
- Comment the code where nontrivial crystallographic operations are performed.

Before calling done:
- Execute generate_structures.py and setup_jobs.py.
- Verify it completes without errors.
- Read at least one intermediate CIF and one quantum espresso input file to confirm it appears reasonable.

In the done summary, include:
- The Materials Project IDs used for the endpoint structures.
- An outline of how the new python workflow works.
- Any scientific assumptions made and a quick justification.
- Any details that might require further expert domain knowledge.
"""

In [9]:
final_state = run_agent(TASK, model, tokenizer, verbose=True)

print("\nFINAL STATE:\n", json.dumps(final_state.to_dict(), indent=2))


=== STEP 1 ===
RAW MODEL OUTPUT:
 { "tool": "read_file", "args": { "path": "generate_structures.py" } }
PARSED TOOL CALL:
 {'tool': 'read_file', 'args': {'path': 'generate_structures.py'}}
TOOL RESULT:
 {'path': 'generate_structures.py', 'content': 'import os\nimport math\nimport numpy as np\n\nfrom pymatgen.core import Structure, Lattice\nfrom pymatgen.io.cif import CifWriter\n\n# SETTINGS\nN_STRUCTURES = 50\n\nBOHR_TO_ANG = 0.52917721092\n\nos.makedirs("structures", exist_ok=True)\n\n# REFERENCE INTERPOLATION\ndef build_structure(par):\n\n    # lattice interpolation (CsPbBr3, from bromideMixBuild.py)\n    cubicScale = 11.09269 * math.sqrt(2)\n\n    cubicA = 1.0\n    cubicB = 1.0\n    cubicC = math.sqrt(2)\n\n    orthoScale = 15.594\n\n    orthoA = 1.0\n    orthoB = 0.9940620455647114\n    orthoC = 1.4220794958797864\n\n    parScale = cubicScale*(1-par) + orthoScale*par\n\n    parA = cubicA*(1-par) + orthoA*par\n    parB = cubicB*(1-par) + orthoB*par\n    parC = cubicC*(1-par) + orth

Retrieving SummaryDoc documents:   0%|          | 0/46 [00:00<?, ?it/s]

TOOL RESULT:
 {'success': True, 'selected_material_id': 'mp-2657', 'candidate_material_ids': ['mp-2657'], 'space_group': {'symbol': 'P4_2/mnm', 'number': 136}, 'output_path': '/nfs/roberts/project/pi_vsb4/byh2/chem_llm/test10/rutile_TiO2.cif'}

=== STEP 4 ===
RAW MODEL OUTPUT:
 { "tool": "read_file", "args": { "path": "setup_jobs.py" } }
PARSED TOOL CALL:
 {'tool': 'read_file', 'args': {'path': 'setup_jobs.py'}}
TOOL RESULT:
 {'path': 'setup_jobs.py', 'content': 'import os\nimport shutil\nfrom pathlib import Path\n\nfrom pymatgen.core import Structure\n\n# SETTINGS\nOVERWRITE = False\n\nTEMPLATE_DIR = "template"\nSTRUCTURE_DIR = "structures"\nCALC_DIR = "calculations"\n\nos.makedirs(CALC_DIR, exist_ok=True)\nos.makedirs(STRUCTURE_DIR, exist_ok=True)\n\n# HELPERS\nANG_TO_BOHR = 1.889726125\n\ndef structure_to_qe(structure, prefix, calculation):\n\n    a, b, c = structure.lattice.abc\n\n    a_bohr = a * ANG_TO_BOHR\n    b_bohr = b * ANG_TO_BOHR\n    c_bohr = c * ANG_TO_BOHR\n\n    celldm

Retrieving SummaryDoc documents:   0%|          | 0/46 [00:00<?, ?it/s]

TOOL RESULT:
 {'success': True, 'selected_material_id': 'mp-2657', 'candidate_material_ids': ['mp-2657'], 'space_group': {'symbol': 'P4_2/mnm', 'number': 136}, 'output_path': '/nfs/roberts/project/pi_vsb4/byh2/chem_llm/test10/rutile_TiO2.cif'}

=== STEP 11 ===
RAW MODEL OUTPUT:
 {
  "tool": "run_python",
  "args": {
    "path": "generate_structures.py"
  }
}
PARSED TOOL CALL:
 {'tool': 'run_python', 'args': {'path': 'generate_structures.py'}}
TOOL RESULT:
 {'stdout': '', 'stderr': 'Traceback (most recent call last):\n  File \x1b"/nfs/roberts/project/pi_vsb4/byh2/chem_llm/test10/generate_structures.py"\x1b, line \x1b46\x1b, in \x1b<module>\x1b\n    \x1bperturbed_structure.frac_coords\x1b = new_frac_coords\n    \x1b^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\x1b\n\x1bAttributeError\x1b: \x1bproperty \'frac_coords\' of \'Structure\' object has no setter\x1b\n'}

=== STEP 12 ===
RAW MODEL OUTPUT:
 {
  "tool": "note",
  "args": {
    "text": "The error in generate_structures.py occurs because Structure